# 1. Chargement des données

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/deep/spam.csv", encoding='latin-1')
print("Shape:", df.shape)
print(df.head())

# Garder les colonnes utiles (texte + label)
df = df[['Class', 'Message']]
df.columns = ['label', 'text']
print("\nDistribution des classes:")
print(df['label'].value_counts())

Shape: (5361, 2)
   Class                                            Message
0      0  Go until jurong point  crazy.. Available only ...
1      0                       Ok lar...Joking wif u oniâ¦
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf he lives arou...

Distribution des classes:
label
0    4616
1     745
Name: count, dtype: int64


# 2. Transformation TF-IDF


Dimension de X (TF-IDF): (5361, 5000)
Dimension de y: (5361,)


 3. Autoencodeur Simple

In [16]:
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Input
from sklearn.model_selection import train_test_split

# 3.4 - Division train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("\nX_train:", X_train.shape, "| X_test:", X_test.shape)

input_dim = X_train.shape[1]



X_train: (4288, 5000) | X_test: (1073, 5000)



# ---- 3.1 & 3.2 : Modèle autoencodeur simple (200 neurones, relu) ----

In [17]:
autoencodeur_simple = Sequential([
    Dense(200, activation='relu', input_shape=(input_dim,)),
    Dense(input_dim, activation='sigmoid')
])
autoencodeur_simple.compile(optimizer='adam', loss='mse')


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


# 3.3 - Architecture

In [25]:
print("\n Architecture Autoencodeur Simple")
autoencodeur_simple.summary()



 Architecture Autoencodeur Simple


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 200)            │     1,000,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5000)           │     1,005,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,015,602 (22.95 MB)

 Trainable params: 2,005,200 (7.65 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 4,010,402 (15.30 MB)

# 3.5 - Entraînement

In [19]:
print("\n--- Entraînement Autoencodeur Simple ---")
autoencodeur_simple.fit(
    X_train, X_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test, X_test),
    verbose=1
)


--- Entraînement Autoencodeur Simple ---
Epoch 1/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - loss: 0.1577 - val_loss: 0.0185
Epoch 2/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 7s 69ms/step - loss: 0.0057 - val_loss: 0.0024
Epoch 3/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - loss: 0.0016 - val_loss: 0.0013
Epoch 4/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 9.2689e-04 - val_loss: 9.0425e-04
Epoch 5/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - loss: 6.5952e-04 - val_loss: 6.9807e-04
Epoch 6/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - loss: 5.2044e-04 - val_loss: 5.7480e-04
Epoch 7/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 4s 62ms/step - loss: 4.3739e-04 - val_loss: 4.9571e-04
Epoch 8/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - loss: 3.8362e-04 - val_loss: 4.4133e-04
Epoch 9/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - loss: 3.4664e-04 - val_loss: 4.0140e-04
Epoch 10/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - loss: 3.1998e-04 - val_loss: 3.7189e-04
Epoch 11/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step

# 3.6 - Évaluation

In [20]:
loss_simple = autoencodeur_simple.evaluate(X_test, X_test, verbose=0)
print(f"\nLoss (MSE) - Autoencodeur Simple (200 neurones, relu) : {loss_simple:.6f}")



Loss (MSE) - Autoencodeur Simple (200 neurones, relu) : 0.000260


# ---- 3.7 : Couche cachée réduite à 10 neurones ----


In [21]:
ae_10 = Sequential([
    Dense(10, activation='relu', input_shape=(input_dim,)),
    Dense(input_dim, activation='sigmoid')
])
ae_10.compile(optimizer='adam', loss='mse')
ae_10.fit(X_train, X_train, epochs=20, batch_size=64,
          validation_data=(X_test, X_test), verbose=0)
loss_10 = ae_10.evaluate(X_test, X_test, verbose=0)
print(f"Loss (MSE) - Autoencodeur Simple (10 neurones, relu)  : {loss_10:.6f}")


Loss (MSE) - Autoencodeur Simple (10 neurones, relu)  : 0.000683


# ---- 3.8 : Activation tanh (200 neurones) ----

In [22]:
ae_tanh = Sequential([
    Dense(200, activation='tanh', input_shape=(input_dim,)),
    Dense(input_dim, activation='sigmoid')
])
ae_tanh.compile(optimizer='adam', loss='mse')
ae_tanh.fit(X_train, X_train, epochs=20, batch_size=64,
            validation_data=(X_test, X_test), verbose=0)
loss_tanh = ae_tanh.evaluate(X_test, X_test, verbose=0)
print(f"Loss (MSE) - Autoencodeur Simple (200 neurones, tanh) : {loss_tanh:.6f}")


Loss (MSE) - Autoencodeur Simple (200 neurones, tanh) : 0.000260


# ---- 3.9 : Extraire représentations compressées (modèle relu 200) ----


In [24]:
encoder_simple = Model(
    inputs=autoencodeur_simple.layers[0].input,
    outputs=autoencodeur_simple.layers[0].output
)
X_train_compressed_simple = encoder_simple.predict(X_train)
X_test_compressed_simple  = encoder_simple.predict(X_test)
print("\nX_train_compressed_simple :", X_train_compressed_simple.shape)
print("X_test_compressed_simple  :", X_test_compressed_simple.shape)


134/134 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step

X_train_compressed_simple : (4288, 200)
X_test_compressed_simple  : (1073, 200)


# 4. Autoencodeur Profond

# 4.1 - Architecture profonde (encodeur + décodeur symétrique)


In [26]:
autoencodeur_deep = Sequential([
    Dense(128, activation='relu', input_shape=(input_dim,)),  # Couche 1
    Dense(64,  activation='relu'),                             # Couche 2
    Dense(32,  activation='relu'),                             # Couche 3  ← goulot
    Dense(64,  activation='relu'),                             # Couche 4
    Dense(128, activation='relu'),                             # Couche 5
    Dense(input_dim, activation='sigmoid')                     # Sortie
])
autoencodeur_deep.compile(optimizer='adam', loss='mse')

print("\n=== Architecture Autoencodeur Profond ===")
autoencodeur_deep.summary()



=== Architecture Autoencodeur Profond ===


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 128)            │       640,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 5000)           │       645,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,305,896 (4.98 MB)

 Trainable params: 1,305,896 (4.98 MB)

 Non-trainable params: 0 (0.00 B)

# 4.2 - Entraînement et comparaison

In [27]:
print("\n--- Entraînement Autoencodeur Profond ---")
autoencodeur_deep.fit(
    X_train, X_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test, X_test),
    verbose=1
)
loss_deep = autoencodeur_deep.evaluate(X_test, X_test, verbose=0)
print(f"\nLoss (MSE) - Autoencodeur Profond  : {loss_deep:.6f}")
print(f"Loss (MSE) - Autoencodeur Simple   : {loss_simple:.6f}")



--- Entraînement Autoencodeur Profond ---
Epoch 1/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - loss: 0.1087 - val_loss: 1.9944e-04
Epoch 2/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - loss: 1.9967e-04 - val_loss: 1.9944e-04
Epoch 3/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 1.9967e-04 - val_loss: 1.9944e-04
Epoch 4/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 1.9967e-04 - val_loss: 1.9944e-04
Epoch 5/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 1.9967e-04 - val_loss: 1.9944e-04
Epoch 6/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - loss: 1.9967e-04 - val_loss: 1.9944e-04
Epoch 7/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - loss: 1.9967e-04 - val_loss: 1.9944e-04
Epoch 8/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 1.9967e-04 - val_loss: 1.9944e-04
Epoch 9/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 1.9967e-04 - val_loss: 1.9944e-04
Epoch 10/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 1.9967e-04 - val_loss: 1.9944e-04
Epoch 11/20
67/67 ━━━━━━━━━━━━

# 4.3 - Représentations compressées (sortie de la couche 32 = index 2)


In [29]:
encoder_deep = Model(
    inputs=autoencodeur_deep.layers[0].input,
    outputs=autoencodeur_deep.layers[2].output   # couche 32 neurones
)
X_train_compressed_deep = encoder_deep.predict(X_train)
X_test_compressed_deep  = encoder_deep.predict(X_test)
print("\nX_train_compressed_deep :", X_train_compressed_deep.shape)
print("X_test_compressed_deep  :", X_test_compressed_deep.shape)


134/134 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step

X_train_compressed_deep : (4288, 32)
X_test_compressed_deep  : (1073, 32)


# 5. Autoencodeur Empilé (Stacked)


# 5.1 - Représentations compressées via encoder_simple


In [30]:
X_train_enc1 = encoder_simple.predict(X_train)
X_test_enc1  = encoder_simple.predict(X_test)
print("\nEntrée autoencodeur 2 - train:", X_train_enc1.shape)


134/134 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

Entrée autoencodeur 2 - train: (4288, 200)


# 5.2 - Deuxième autoencodeur sur les représentations compressées


In [31]:
enc1_dim = X_train_enc1.shape[1]

autoencodeur2 = Sequential([
    Dense(50, activation='relu', input_shape=(enc1_dim,)),
    Dense(enc1_dim, activation='sigmoid')
])
autoencodeur2.compile(optimizer='adam', loss='mse')

print("\n=== Architecture Autoencodeur 2 (Stacked) ===")
autoencodeur2.summary()



=== Architecture Autoencodeur 2 (Stacked) ===


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 50)             │        10,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 200)            │        10,200 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,250 (79.10 KB)

 Trainable params: 20,250 (79.10 KB)

 Non-trainable params: 0 (0.00 B)

# 5.3 & 5.4 - Entraînement et comparaison


In [32]:
print("\n--- Entraînement Autoencodeur Empilé ---")
autoencodeur2.fit(
    X_train_enc1, X_train_enc1,
    epochs=20,
    batch_size=64,
    validation_data=(X_test_enc1, X_test_enc1),
    verbose=1
)
loss_stack = autoencodeur2.evaluate(X_test_enc1, X_test_enc1, verbose=0)
print(f"\n=== Comparaison des performances (MSE) ===")
print(f"Autoencodeur Simple  : {loss_simple:.6f}")
print(f"Autoencodeur Profond : {loss_deep:.6f}")
print(f"Autoencodeur Empilé  : {loss_stack:.6f}")



--- Entraînement Autoencodeur Empilé ---
Epoch 1/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0143 - val_loss: 0.0082
Epoch 2/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0070 - val_loss: 0.0079
Epoch 3/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0067 - val_loss: 0.0076
Epoch 4/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0063 - val_loss: 0.0070
Epoch 5/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0059 - val_loss: 0.0065
Epoch 6/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0054 - val_loss: 0.0058
Epoch 7/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0047 - val_loss: 0.0049
Epoch 8/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0040 - val_loss: 0.0039
Epoch 9/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0030 - val_loss: 0.0029
Epoch 10/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0022 - val_loss: 0.0021
Epoch 11/20
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0016 - val_loss: 0.0015
Epoch 12/20
67/67 ━━━━━━━━━━━━━━

# 5.5 - Extraire représentations compressées finales


In [34]:
encoder_stack = Model(
    inputs=autoencodeur2.layers[0].input,
    outputs=autoencodeur2.layers[0].output
)
X_train_compressed_stack = encoder_stack.predict(X_train_enc1)
X_test_compressed_stack  = encoder_stack.predict(X_test_enc1)
print("\nX_train_compressed_stack :", X_train_compressed_stack.shape)
print("X_test_compressed_stack  :", X_test_compressed_stack.shape)


134/134 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

X_train_compressed_stack : (4288, 50)
X_test_compressed_stack  : (1073, 50)


# 5.6 & 5.7 : Classification MLP sur les représentations


In [38]:
from tensorflow.keras.layers import Dropout, Input
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.models import Model # Import Model for Functional API

def build_mlp_classifier(input_dim):
    # Define the input layer
    input_tensor = Input(shape=(input_dim,))

    # Build the MLP layers
    x = Dense(64, activation='relu')(input_tensor)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    output_tensor = Dense(1, activation='sigmoid')(x)

    # Create the model using the Functional API
    model = Model(inputs=input_tensor, outputs=output_tensor)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

results = {}

# --- MLP sur représentations simples ---
mlp_simple = build_mlp_classifier(X_train_compressed_simple.shape[1])
mlp_simple.fit(X_train_compressed_simple, y_train,
               epochs=20, batch_size=64,
               validation_data=(X_test_compressed_simple, y_test),
               verbose=0)
y_pred_simple = (mlp_simple.predict(X_test_compressed_simple) > 0.5).astype(int).flatten()
acc_simple = accuracy_score(y_test, y_pred_simple)
results['Simple'] = acc_simple

# --- MLP sur représentations deep ---
mlp_deep = build_mlp_classifier(X_train_compressed_deep.shape[1])
mlp_deep.fit(X_train_compressed_deep, y_train,
             epochs=20, batch_size=64,
             validation_data=(X_test_compressed_deep, y_test),
             verbose=0)
y_pred_deep = (mlp_deep.predict(X_test_compressed_deep) > 0.5).astype(int).flatten()
acc_deep = accuracy_score(y_test, y_pred_deep)
results['Deep'] = acc_deep

# --- MLP sur représentations stacked ---
mlp_stack = build_mlp_classifier(X_train_compressed_stack.shape[1])
mlp_stack.fit(X_train_compressed_stack, y_train,
              epochs=20, batch_size=64,
              validation_data=(X_test_compressed_stack, y_test),
              verbose=0)
y_pred_stack = (mlp_stack.predict(X_test_compressed_stack) > 0.5).astype(int).flatten()
acc_stack = accuracy_score(y_test, y_pred_stack)
results['Stacked'] = acc_stack

# --- Résultats finaux ---
print("\n" + "="*55)
print("  RÉSULTATS FINAUX - Classification MLP")
print("="*55)
print(f"{'Représentation':<20} {'Accuracy':>10}")
print("-"*35)
for k, v in results.items():
    print(f"{k:<20} {v*100:>9.2f}%")
print("="*55)

print("\n--- Rapport détaillé : Autoencodeur Simple ---")
print(classification_report(y_test, y_pred_simple, target_names=['ham', 'spam']))

print("\n--- Rapport détaillé : Autoencodeur Profond ---")
print(classification_report(y_test, y_pred_deep, target_names=['ham', 'spam']))

print("\n--- Rapport détaillé : Autoencodeur Empilé ---")
print(classification_report(y_test, y_pred_stack, target_names=['ham', 'spam']))


34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

  RÉSULTATS FINAUX - Classification MLP
Représentation         Accuracy
-----------------------------------
Simple                  100.00%
Deep                    100.00%
Stacked                 100.00%

--- Rapport détaillé : Autoencodeur Simple ---
